<a href="https://colab.research.google.com/github/Gerardomedinav/crime-hotspot-predictor/blob/main/notebooks/entrega_Final_delitos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Proyecto Integrador: Inteligencia Geoespacial en Seguridad Urbana
## Análisis de Delitos en la Ciudad de Buenos Aires

**Dataset:** [Delitos 2024 – Gobierno de la Ciudad de Buenos Aires](https://data.buenosaires.gob.ar/de/dataset/delitos/resource/49f58c2e-21d7-4766-84e0-4bb753d28478)

## Introducción y Objetivos

El presente proyecto tiene como objetivo analizar el comportamiento delictivo en la Ciudad de Buenos Aires a partir del estudio de datos geoespaciales y temporales. Se busca comprender cómo se distribuyen los delitos en el territorio y en el tiempo, identificando patrones que permitan explicar su ocurrencia y sentar las bases para futuros modelos predictivos.

En particular, el trabajo se propone:

- Comprender la distribución de los delitos en la ciudad  
- Identificar patrones geoespaciales y zonas de riesgo (hotspots)  
- Detectar comportamientos temporales según horas, días y meses  
- Preparar y transformar los datos para su uso en modelos de Machine Learning  

Para alcanzar estos objetivos, se desarrolla un proceso completo de análisis de datos que incluye limpieza, validación, exploración y transformación de variables. El proyecto parte de un dataset con múltiples dimensiones, que combina información geográfica, temporal y categórica, lo que permite abordar el problema desde un enfoque integral.

El trabajo comienza con un análisis exploratorio de datos (EDA), etapa fundamental para comprender la estructura del dataset, detectar inconsistencias y descubrir patrones iniciales. A partir de este análisis, se aplican técnicas de ingeniería de características (Feature Engineering) para mejorar la representación de la información, incorporando variables que capturan el contexto temporal, la frecuencia de ocurrencia y la dinámica espacial del delito.

Finalmente, se obtiene un dataset optimizado que permite modelar el fenómeno delictivo como un problema de Machine Learning, facilitando la identificación de contextos de mayor riesgo y contribuyendo al desarrollo de herramientas analíticas aplicadas a la seguridad urbana.


## Importación de librerías

A continuación se cargan las librerías necesarias para el análisis de datos, la visualización y el modelado.

- pandas: manipulación de datos
- numpy: cálculos numéricos
- matplotlib y seaborn: visualización
- folium: mapas geoespaciales




In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import folium
from folium.plugins import HeatMap

## Carga del dataset

Se carga el dataset de delitos correspondiente al año 2024.

En este paso se realiza una primera inspección general para entender:

- cantidad de registros
- columnas disponibles
- estructura de los datos


In [ ]:
df = pd.read_csv("delitos_2024.csv", encoding="latin1", engine="python", on_bad_lines="warn")

print("Dimensiones del dataset:", df.shape)
df.head()

## Exploración inicial

Antes de realizar cualquier transformación, es importante analizar:

- tipos de datos
- valores nulos
- consistencia general

Esto permite detectar problemas que podrían afectar el modelo.


In [ ]:
df.info()


## Análisis inicial del dataset

El dataset contiene un total de **158.838 registros** y **15 columnas**, donde cada fila representa un evento delictivo registrado en la Ciudad de Buenos Aires.

Se observan tres tipos principales de variables:

- **Variables numéricas**: como `anio`, `cantidad`, `latitud`, `longitud`
- **Variables categóricas**: como `tipo`, `subtipo`, `barrio`, `uso_arma`
- **Variables temporales**: como `fecha`, `mes`, `dia`, `franja`

Esto indica que el dataset contiene información suficiente para realizar un análisis tanto **temporal como geoespacial** del delito.

In [ ]:
df.isnull().sum()

## Análisis de valores nulos

Se identifican valores faltantes en algunas variables clave del dataset:

- `franja`: 47 valores nulos
- `barrio` y `comuna`: aproximadamente 3900 valores nulos
- `latitud` y `longitud`: aproximadamente 3800 valores nulos

Es importante destacar que:

- La variable `fecha` no presenta valores nulos, lo cual es positivo para el análisis temporal.
- Sin embargo, las variables geoespaciales (`latitud`, `longitud`) sí presentan valores faltantes, lo que puede afectar directamente el análisis de mapas y la detección de zonas calientes.

Por este motivo, se procede a eliminar registros incompletos en estas variables críticas.

## Limpieza de datos

En esta etapa se eliminan o corrigen valores inválidos para garantizar la calidad del dataset.

Se aplican los siguientes pasos:

- Conversión de fechas
- Eliminación de valores nulos críticos
- Validación de rango horario

In [ ]:
# Convertimos la fecha al formato datetime
df['fecha'] = pd.to_datetime(df['fecha'], errors='coerce')

# Eliminamos valores nulos importantes
df = df.dropna(subset=['fecha', 'latitud', 'longitud'])

# Convertimos la franja horaria a numérico
df['franja'] = pd.to_numeric(df['franja'], errors='coerce')

df = df.dropna(subset=['franja'])

# Validamos rango de horas
df = df[(df['franja'] >= 0) & (df['franja'] <= 23)]

print("Registros luego de limpieza:", df.shape)


## Limpieza de datos

Luego del análisis inicial, se realizó un proceso de limpieza del dataset para garantizar la calidad de los datos.

Se aplicaron las siguientes transformaciones:

- Se convirtió la columna `fecha` al tipo datetime.
- Se eliminaron registros con valores nulos en variables críticas como:
  - `fecha`
  - `latitud`
  - `longitud`
- Se transformó la variable `franja` a formato numérico.
- Se eliminaron registros con valores nulos en la franja horaria.
- Se filtraron únicamente valores válidos de hora dentro del rango 0 a 23.

Como resultado, el dataset pasó de **158.838 registros a 154.963 registros**, eliminando alrededor de **3.875 registros (aprox. 2.44%)**.

Este proceso permite asegurar que el análisis geoespacial y el modelado se realicen sobre datos completos y consistentes.


## Análisis de distribuciones

Se analizan las variables numéricas para comprender:

- cómo se distribuyen los datos
- posibles concentraciones
- valores atípicos

Esto ayuda a entender el comportamiento del fenómeno delictivo.

In [ ]:
# =========================
# ANÁLISIS TEMPORAL: DELITOS POR MES
# =========================

#  Asegurar que fecha es datetime
df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")
df["mes"] = df["fecha"].dt.month


#  Conteo mensual
conteo_mensual = df.groupby("mes").size().sort_index()

#  Gráfico
plt.figure(figsize=(10,5))
plt.plot(conteo_mensual.index, conteo_mensual.values, marker='o')

# Etiquetas
for x, y in zip(conteo_mensual.index, conteo_mensual.values):
    plt.text(x, y, str(y), ha='center', va='bottom')

plt.title("Cantidad de delitos por mes")
plt.xlabel("Mes")
plt.ylabel("Cantidad de delitos")

#  Mostrar solo meses existentes
plt.xticks(conteo_mensual.index)

plt.grid()
plt.tight_layout()
plt.show()

##  Análisis Temporal de Delitos por Mes

El análisis muestra una variación moderada en la cantidad de delitos a lo largo del año, con un pico en marzo (13.554 casos) y un mínimo en septiembre (11.981 casos). Se observa una tendencia de disminución hacia mitad de año y una posterior recuperación en los últimos meses, lo que sugiere la presencia de un comportamiento estacional en la actividad delictiva.

In [ ]:
# =========================
# LISTADO DE TIPOS Y SUBTIPOS (EDA)
# =========================

listado_delitos = (
    df.groupby(['tipo', 'subtipo'])
    .size()
    .reset_index(name='frecuencia')
    .sort_values(by="frecuencia", ascending=False)
)

print("Top 10 delitos:")
print(listado_delitos.head(10))


# =========================
# VISUALIZACIÓN
# =========================

plt.figure(figsize=(8,5))
sns.barplot(
    data=listado_delitos.head(10),
    x='frecuencia',
    y='subtipo'
)
plt.title("Top 10 delitos más frecuentes")
plt.xlabel("Frecuencia")
plt.ylabel("Tipo de delito")
plt.show()


# =========================
# FEATURE ENGINEERING
# =========================

df['fecha'] = pd.to_datetime(df['fecha'], errors='coerce')
df['dia_semana'] = df['fecha'].dt.dayofweek
df['es_fin_de_semana'] = (df['dia_semana'] >= 5).astype(int)


freq_barrio = df.groupby('barrio').size()
df['densidad_barrio'] = df['barrio'].map(freq_barrio)

freq_hora = df.groupby('franja').size()
df['densidad_hora'] = df['franja'].map(freq_hora)

freq_dia = df.groupby('dia_semana').size()
df['densidad_dia'] = df['dia_semana'].map(freq_dia)

freq_finde = df.groupby('es_fin_de_semana').size()
df['densidad_finde'] = df['es_fin_de_semana'].map(freq_finde)
#Estas variables fueron utilizadas para el análisis exploratorio, pero no forman parte del modelo final.
print("✔ Features de densidad creadas correctamente")



## Análisis de Tipos de Delitos

El análisis muestra que los delitos más frecuentes corresponden a "Robo total" y "Hurto total", concentrando la mayor parte de los casos. Esto indica que la criminalidad en el dataset está dominada por delitos contra la propiedad, mientras que delitos de mayor gravedad presentan una baja

In [ ]:
# =========================
# 8 ANÁLISIS: DELITOS POR BARRIO
# =========================

# Top 10 barrios con mayor cantidad de delitos
top_barrios = df['barrio'].value_counts().head(10)

plt.figure(figsize=(10,6))

sns.barplot(
    x=top_barrios.values,
    y=top_barrios.index
)

plt.title("Top 10 Barrios con Mayor Cantidad de Delitos (2024)")
plt.xlabel("Cantidad de Incidentes")
plt.ylabel("Barrio")

# Etiquetas numéricas
for i, v in enumerate(top_barrios.values):
    plt.text(v, i, f'{v}', va='center')

plt.tight_layout()
plt.show()

## Análisis de Delitos por Barrio

El gráfico muestra que la distribución de delitos es desigual entre los barrios, con Palermo concentrando la mayor cantidad de incidentes (12.789), seguido por Balvanera y Flores. Esto indica la existencia de zonas con mayor incidencia delictiva, lo que refuerza la importancia del análisis geoespacial para identificar áreas críticas de seguridad.

In [ ]:
# =========================
# 9 ANÁLISIS: DELITOS POR FRANJA HORARIA
# =========================

# Ordenar horas correctamente
orden_horas = sorted(df['franja'].dropna().unique())

# Contar valores y ordenar
conteo_horas = df['franja'].value_counts().sort_index()

plt.figure(figsize=(12,6))

sns.countplot(
    x='franja',
    data=df,
    order=orden_horas,
    color='steelblue'
)

plt.title("Frecuencia de Delitos por Franja Horaria")
plt.xlabel("Hora del día")
plt.ylabel("Cantidad de Incidentes")

# Etiquetas numéricas más alineadas
for i, v in enumerate(conteo_horas.values):
    plt.text(i, v + 50, str(v), ha='center')  # pequeño offset

# Grilla para mejor lectura
plt.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## Análisis de Delitos por Franja Horaria

El análisis evidencia que la actividad delictiva no es uniforme a lo largo del día. Se observa una menor cantidad de delitos durante la madrugada (entre las 2:00 y 5:00), con valores mínimos cercanos a los 2.800 casos. A partir de la mañana, la frecuencia aumenta progresivamente, alcanzando un primer pico hacia las 8:00 (8.068 incidencias). Sin embargo, el nivel más alto se registra en la tarde-noche, especialmente entre las 17:00 y 20:00 horas, donde se concentran los máximos valores (hasta 8.704 casos a las 18:00).  

Esto sugiere que la mayor actividad delictiva coincide con los horarios de mayor circulación y actividad urbana, mientras que las horas de menor movimiento presentan menor incidencia.


In [ ]:
# =========================
#  RELACIÓN DÍA vs HORA
# =========================

# Asegurar formato datetime
df['fecha'] = pd.to_datetime(df['fecha'], errors='coerce')

# Crear días de la semana
df['dia_semana'] = df['fecha'].dt.dayofweek

mapa_dias = {
    0: 'LUNES',
    1: 'MARTES',
    2: 'MIERCOLES',
    3: 'JUEVES',
    4: 'VIERNES',
    5: 'SABADO',
    6: 'DOMINGO'
}

df['dia_nombre'] = df['dia_semana'].map(mapa_dias)

# Limpiar datos necesarios
df_clean = df.dropna(subset=['franja', 'dia_nombre'])

# =========================
# TABLA RELACIÓN (FORMA SEGURA)
# =========================

tabla_relacion = pd.crosstab(
    df_clean['dia_nombre'],
    df_clean['franja']
)

# Orden correcto de días
orden_dias = [
    "LUNES", "MARTES", "MIERCOLES",
    "JUEVES", "VIERNES", "SABADO", "DOMINGO"
]

tabla_relacion = tabla_relacion.reindex(orden_dias)

# Ordenar horas
tabla_relacion = tabla_relacion.reindex(
    sorted(tabla_relacion.columns),
    axis=1
)

# =========================
# HEATMAP
# =========================

plt.figure(figsize=(15,6))

sns.heatmap(
    tabla_relacion,
    cmap="YlOrRd",
    linewidths=0.3
)

plt.title("Mapa de Calor: Día vs Franja Horaria")
plt.xlabel("Hora del día")
plt.ylabel("Día de la semana")

plt.tight_layout()
plt.show()


## Relación entre Día de la Semana y Franja Horaria

El mapa de calor muestra que la actividad delictiva presenta un patrón consistente a lo largo de la semana, con menor incidencia durante la madrugada (entre las 1:00 y 5:00) y un aumento progresivo a partir de la mañana. Se destacan picos de mayor concentración entre las 7:00 y 9:00, y principalmente en la franja de la tarde-noche (entre las 17:00 y 21:00), donde se observan los valores más altos en casi todos los días.

Asimismo, se identifica que los días viernes y sábado presentan una mayor intensidad delictiva en horas nocturnas en comparación con el resto de la semana, lo que sugiere una relación con la mayor actividad social y movilidad urbana en esos períodos. En general, el comportamiento es bastante uniforme entre días, pero con un claro refuerzo de la actividad en horarios de mayor circulación.


In [ ]:
df[['latitud', 'longitud', 'franja']].hist(bins=40, figsize=(12,6))
plt.show()

## Análisis de distribuciones y validación geoespacial

Al analizar las variables latitud y longitud, se detectaron valores extremadamente grandes que no corresponden a coordenadas reales de la Ciudad de Buenos Aires.

Esto indica la presencia de datos corruptos o errores de carga en el dataset, los cuales afectan directamente la visualización y el análisis geoespacial.

Para solucionar este problema, se aplica un filtro geográfico basado en los límites aproximados de CABA.


In [ ]:
# =========================
# LIMPIEZA GEOESPACIAL (CRÍTICA)
# =========================

df = df[
    (df['latitud'].between(-34.8, -34.4)) &
    (df['longitud'].between(-58.6, -58.3))
]

print("Registros luego de limpieza geográfica:", df.shape)


Para garantizar la calidad del análisis, se aplicó un filtro geográfico basado en los límites aproximados de la ciudad:

- Latitud entre -34.8 y -34.4  
- Longitud entre -58.6 y -58.3  

Como resultado, el dataset se redujo a **154.879 registros**, eliminando únicamente aquellos casos que no aportan valor al análisis geoespacial.

Este proceso permite:

- eliminar ruido en los datos  
- asegurar coherencia espacial  
- mejorar la calidad de las visualizaciones y del modelo  

En problemas de inteligencia geoespacial, este tipo de limpieza es fundamental, ya que coordenadas incorrectas pueden generar patrones falsos o conclusiones erróneas sobre zonas de riesgo.


La proporción de registros eliminados es baja respecto al total del dataset, lo que indica que la limpieza no afecta la representatividad de los datos, sino que mejora su calidad.


In [ ]:
df[['latitud', 'longitud', 'franja']].hist(bins=40, figsize=(12,6))
plt.show()

### Interpretación después de la limpieza

Luego de aplicar el filtro geográfico, las distribuciones de latitud y longitud muestran rangos coherentes con la ubicación de la Ciudad de Buenos Aires.

Se elimina la distorsión provocada por valores extremos, permitiendo un análisis más preciso de la concentración delictiva.

La variable franja mantiene su comportamiento, lo cual confirma que no presentaba problemas estructurales.

Esto valida que el dataset ahora es adecuado para análisis geoespacial y modelado.
``

## Análisis temporal dinámico

Se construye un mapa interactivo que permite analizar cómo varía la distribución del delito según el mes.

In [ ]:
# ==========================================================
# VISUALIZACIÓN GEOESPACIAL INTERACTIVA
# ==========================================================

import folium
from folium.plugins import HeatMap, MarkerCluster
import ipywidgets as widgets
from IPython.display import display

# ==========================================================
# 1. PREPARACIÓN DE DATOS
# ==========================================================

# Limpiar datos
df_mapa = df.dropna(subset=["latitud", "longitud"]).copy()

# Filtro geográfico (CABA)
df_mapa = df_mapa[
    (df_mapa['latitud'].between(-34.70, -34.53)) &
    (df_mapa['longitud'].between(-58.53, -58.34))
]

# Asegurar variables temporales
df_mapa['fecha'] = pd.to_datetime(df_mapa['fecha'], errors='coerce')
df_mapa['mes'] = df_mapa['fecha'].dt.month

# ==========================================================
# LISTAS COMPLETAS
# ==========================================================

# Meses + opción total
lista_meses = ["Todos"] + list(range(1, 13))

# Barrios + opción total
lista_barrios = sorted(df_mapa["barrio"].dropna().unique())
lista_barrios = ["Todos"] + lista_barrios

# ==========================================================
# FUNCIÓN DEL MAPA
# ==========================================================

def generar_mapa_avanzado(mes_seleccionado, barrio_seleccionado, tipo_visualizacion):

    df_filtrado = df_mapa.copy()

    # ==============================
    # FILTROS
    # ==============================

    # Filtro por mes (solo si no es "Todos")
    if mes_seleccionado != "Todos":
        df_filtrado = df_filtrado[df_filtrado["mes"] == mes_seleccionado]

    # Filtro por barrio
    if barrio_seleccionado != "Todos":
        df_filtrado = df_filtrado[df_filtrado["barrio"] == barrio_seleccionado]

    # Validación
    if df_filtrado.empty:
        print("No hay datos para los filtros seleccionados")
        return

    # ==============================
    # INFO RÁPIDA
    # ==============================

    mes_texto = "Todos los meses" if mes_seleccionado == "Todos" else mes_seleccionado

    print("ANÁLISIS")
    print(f"Total de delitos: {len(df_filtrado)}")
    print(f"Mes: {mes_texto}")
    print(f"Barrio: {barrio_seleccionado}")

    if barrio_seleccionado == "Todos":
        print("\nTop barrios:")
        print(df_filtrado["barrio"].value_counts().head(3))

    print("-" * 30)

    # ==============================
    # MAPA
    # ==============================

    m = folium.Map(
        location=[-34.61, -58.44],
        zoom_start=12,
        tiles="cartodbpositron"
    )

    # ==============================
    # HEATMAP
    # ==============================

    if tipo_visualizacion == 'Mapa de Calor':

        HeatMap(
            df_filtrado[["latitud", "longitud"]].values.tolist(),
            radius=12,
            blur=18
        ).add_to(m)

    # ==============================
    # CLUSTERS
    # ==============================

    else:
        marker_cluster = MarkerCluster().add_to(m)

        muestra = df_filtrado.sample(min(len(df_filtrado), 1000))

        for _, row in muestra.iterrows():

            folium.CircleMarker(
                location=[row["latitud"], row["longitud"]],
                radius=4,
                color="red",
                fill=True,
                fill_opacity=0.6,
                popup=f"""
                Barrio: {row.get('barrio','N/A')}<br>
                Delito: {row.get('tipo','N/A')}<br>
                Fecha: {row.get('fecha','N/A')}
                """
            ).add_to(marker_cluster)

    display(m)

# ==========================================================
# INTERFAZ INTERACTIVA
# ==========================================================

widgets.interact(
    generar_mapa_avanzado,

    mes_seleccionado=widgets.Dropdown(
        options=lista_meses,
        value="Todos",  # DEFAULT: TODO EL AÑO
        description='Mes:'
    ),

    barrio_seleccionado=widgets.Dropdown(
        options=lista_barrios,
        value="Todos",
        description='Barrio:'
    ),

    tipo_visualizacion=[
        'Mapa de Calor',
        'Clusters'
    ]
)


## Análisis Geoespacial Global (Todos los Meses)

Al considerar la totalidad del período analizado, se registran 154.737 delitos, lo que permite observar una visión completa de la distribución espacial del fenómeno delictivo en la ciudad.

Los datos muestran una concentración significativa en determinados barrios, destacándose Palermo (12.789 casos), Balvanera (9.966) y Flores (8.508), lo que evidencia que la actividad delictiva no es homogénea y se concentra en zonas específicas.

El mapa de calor refuerza este comportamiento, mostrando una mayor densidad en áreas centrales y de alta circulación urbana, mientras que otras zonas presentan menor intensidad. Esta distribución sugiere una fuerte relación entre la ocurrencia de delitos y factores como densidad poblacional, actividad comercial y movimiento diario de personas.

En conjunto, el análisis confirma la existencia de hotspots bien definidos, lo que resulta clave para la planificación de estrategias de prevención y asignación de recursos de seguridad.

## Conclusión del EDA

El análisis exploratorio permitió identificar patrones claros en la distribución delictiva tanto en el espacio como en el tiempo. A nivel geoespacial, se observó una fuerte concentración de delitos en determinados barrios, destacándose Palermo, Balvanera y Flores, lo que evidencia la presencia de zonas críticas asociadas a alta densidad urbana y circulación de personas.

En cuanto a la dimensión temporal, se detectaron patrones relevantes según la franja horaria y el día de la semana. La actividad delictiva muestra un aumento progresivo desde la mañana y alcanza sus picos en la tarde-noche, especialmente en horarios de mayor movilidad. Asimismo, existen diferencias entre días laborales y fines de semana, lo que sugiere una relación con dinámicas sociales y urbanas.

El análisis combinado (día vs hora) confirmó la existencia de patrones consistentes a lo largo del tiempo, mientras que el estudio global por meses permitió observar que estas concentraciones se mantienen de forma estable, reforzando la existencia de zonas de alta incidencia delictiva (hotspots).

Estos hallazgos justifican la necesidad de transformar variables y aplicar técnicas de Feature Engineering para capturar correctamente estos comportamientos, lo cual resulta fundamental para el desarrollo de modelos de Machine Learning en etapas posteriores.


## Feature Engineering

Luego de realizar el análisis exploratorio (EDA), observamos que el fenómeno delictivo presenta:

- patrones espaciales (zonas de concentración)
- patrones temporales (hora, día, mes)

Sin embargo, estas variables no siempre están en una forma directamente interpretable por los modelos de Machine Learning.

Por este motivo, se aplica **Feature Engineering**, que consiste en:

- transformar variables existentes
- crear nuevas variables
- mejorar la representación de los datos

El objetivo es facilitar el aprendizaje del modelo y mejorar su capacidad predictiva.

## Creación de variables temporales

A partir de la variable fecha, se generan nuevas variables que capturan el contexto temporal del delito.

Esto permite al modelo detectar patrones como:

- mayor criminalidad en ciertos meses
- diferencias entre días laborales y fines de semana

In [ ]:
# ==========================================================
# 1. VARIABLES TEMPORALES
# ==========================================================

import pandas as pd
import numpy as np

df['fecha'] = pd.to_datetime(df['fecha'], errors='coerce')

df['mes'] = df['fecha'].dt.month
df['dia_semana'] = df['fecha'].dt.dayofweek
df['es_fin_de_semana'] = (df['dia_semana'] >= 5).astype(int)

# ==========================================================
# 2. HORA CÍCLICA
# ==========================================================

df['hora_sin'] = np.sin(2 * np.pi * df['franja'] / 24)
df['hora_cos'] = np.cos(2 * np.pi * df['franja'] / 24)

# ==========================================================
# 3. ZONA GEOESPACIAL
# ==========================================================

df['zona_lat'] = (df['latitud'] * 100).astype(int)
df['zona_long'] = (df['longitud'] * 100).astype(int)
df['zona'] = df['zona_lat'].astype(str) + "_" + df['zona_long'].astype(str)

# ==========================================================
# 4. FEATURES
# ==========================================================

features = [
    'latitud',
    'longitud',
    'hora_sin',
    'hora_cos',
    'mes',
    'dia_semana',
    'es_fin_de_semana',
    'zona'
]

X = df[features].copy()

# ==========================================================
# 5. SPLIT (SIN zona_caliente)
# ==========================================================

from sklearn.model_selection import train_test_split

X_train, X_test = train_test_split(
    X,
    test_size=0.2,
    random_state=42
)

# ==========================================================
# 6. TARGET SIN LEAKAGE
# ==========================================================

conteo_zona_train = X_train.groupby('zona').size()

X_train['densidad_zona'] = X_train['zona'].map(conteo_zona_train)

X_test['densidad_zona'] = X_test['zona'].map(conteo_zona_train)
X_test['densidad_zona'] = X_test['densidad_zona'].fillna(0)

umbral = X_train['densidad_zona'].quantile(0.75)

y_train = (X_train['densidad_zona'] >= umbral).astype(int)
y_test = (X_test['densidad_zona'] >= umbral).astype(int)

# ==========================================================
# 7. LIMPIAR
# ==========================================================

X_train = X_train.drop(columns=['zona', 'densidad_zona'])
X_test = X_test.drop(columns=['zona', 'densidad_zona'])

# ==========================================================
# 8. ESCALADO
# ==========================================================

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train[['latitud', 'longitud']] = scaler.fit_transform(
    X_train[['latitud', 'longitud']]
)

X_test[['latitud', 'longitud']] = scaler.transform(
    X_test[['latitud', 'longitud']]
)

# Renombrar
X_train = X_train.rename(columns={
    'latitud': 'lat_std',
    'longitud': 'long_std'
})

X_test = X_test.rename(columns={
    'latitud': 'lat_std',
    'longitud': 'long_std'
})

# ==========================================================
# 9. VERIFICACIÓN
# ==========================================================

print("PIPELINE OK")

print("\nFeatures finales:")
print(X_train.columns)

print("\nDistribución target:")
print(y_train.value_counts())

print("\nPrimeras filas:")
print(X_train.head())

### Construcción del dataset y definición del target

En este pipeline se construye el conjunto de datos final para el modelado, combinando información **geoespacial** y **temporal**.

Las variables temporales (`mes`, `dia_semana`, `es_fin_de_semana`) y la representación cíclica de la hora (`hora_sin`, `hora_cos`) describen el **contexto en el que ocurre cada evento**, mientras que la ubicación se representa mediante coordenadas estandarizadas (`lat_std`, `long_std`).

El objetivo del modelo es identificar si un evento ocurre en una **zona caliente (hotspot espacial)**. Para ello, se construye una grilla geográfica (`zona`) y se calcula la densidad de incidentes **exclusivamente con el conjunto de entrenamiento**, utilizando el percentil 75 como umbral. Este enfoque evita fuga de información (*data leakage*) y mantiene la coherencia del target.

De este modo, el modelo no predice únicamente el momento del delito, sino la **probabilidad de que un evento, dado su contexto temporal y ubicación, pertenezca a una zona de alta concentración delictiva**.

## Análisis por día de la semana

Antes de avanzar, se analiza si existen diferencias en la cantidad de delitos según el día.

In [ ]:
# =========================
# ANÁLISIS: DELITOS POR DÍA DE LA SEMANA
# =========================

# Conteo por día de la semana (0=Lunes, 6=Domingo)
conteo_dias = df['dia_semana'].value_counts().sort_index()

# Nombres de los días
dias_nombre = ['L', 'M', 'Mi', 'J', 'V', 'S', 'D']

plt.figure(figsize=(8,5))

# Gráfico de barras
plt.bar(dias_nombre, conteo_dias.values, color='steelblue')

# Títulos
plt.title("Cantidad de delitos por día de la semana")
plt.xlabel("Día")
plt.ylabel("Cantidad")

# Etiquetas encima de cada barra
for i, v in enumerate(conteo_dias.values):
    plt.text(i, v + 50, str(v), ha='center')

# Grilla para mejor lectura
plt.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

Se observan diferencias en la frecuencia de delitos entre los distintos días.

Esto confirma que la variable temporal es relevante y debe ser incorporada en el modelo.

## Transformación cíclica de la hora

La variable 'franja' representa horas de 0 a 23.

El problema es que:
- 23 y 0 están muy cerca en la realidad
- pero numéricamente parecen alejados

Para resolver esto, se utiliza una transformación trigonométrica.


In [ ]:
# =========================
# TRANSFORMACIÓN CÍCLICA
# =========================

import numpy as np
df.loc[:, 'hora_sin'] = np.sin(2 * np.pi * df['franja'] / 24)
df.loc[:, 'hora_cos'] = np.cos(2 * np.pi * df['franja'] / 24)


# =========================
# COMPARACIÓN DE ESCALA (ML)
# =========================

import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12,5))

# -------------------------
# ANTES (variable original)
# -------------------------
plt.subplot(1,2,1)

sns.histplot(df['franja'], bins=24, kde=False)

plt.title("Antes: Variable Hora (Escala Original)")
plt.xlabel("Hora (0–23)")
plt.ylabel("Frecuencia")

# -------------------------
# DESPUÉS (variables transformadas)
# -------------------------
plt.subplot(1,2,2)

sns.histplot(df['hora_sin'], bins=24, color='blue', label='sin', alpha=0.6)
sns.histplot(df['hora_cos'], bins=24, color='orange', label='cos', alpha=0.6)

plt.title("Después: Variables Transformadas (-1 a 1)")
plt.xlabel("Valor")
plt.ylabel("Frecuencia")
plt.legend()

plt.tight_layout()
plt.show()


Con esta transformación:

- las 23:00 y 00:00 quedan cercanas en el espacio matemático
- se preserva la naturaleza circular del tiempo

Esto mejora significativamente el rendimiento de modelos lineales.
La variable original "franja" presenta una escala amplia (0 a 23), lo que puede generar dificultades para los modelos de Machine Learning al interpretar distancias entre valores. Tras aplicar la transformación trigonométrica, las nuevas variables (hora_sin y hora_cos) quedan acotadas en un rango entre -1 y 1, lo que permite una representación más uniforme y comparable con otras variables. Esta normalización mejora la estabilidad del modelo y evita interpretaciones erróneas asociadas a la escala de la variable original.

## Escalado de variables

Las variables tienen distintas escalas:

- coordenadas → valores continuos
- variables binarias → 0 y 1

Esto puede afectar el modelo.

Por eso se aplica StandardScaler.

In [ ]:
# ==========================================================
# ANÁLISIS DE CORRELACIÓN
# ==========================================================

# quitar columnas no numéricas
X_corr = X_train.select_dtypes(include=['number'])

corr = X_corr.corr()

plt.figure(figsize=(12, 7))

sns.heatmap(
    corr,
    annot=True,
    cmap='coolwarm',
    fmt=".2f",
    linewidths=0.5
)

plt.title("Matriz de Correlación de Variables Predictoras")
plt.tight_layout()
plt.show()

# ==========================================================
# CORRELACIÓN CON TARGET
# ==========================================================

df_model = X_train.copy()
df_model['zona_caliente'] = y_train

df_model = df_model.select_dtypes(include=['number'])

corr_target = df_model.corr()

corr_ordenada = corr_target['zona_caliente'].abs().sort_values(ascending=False)

print("Variables ordenadas por correlación con el target:")
print(corr_ordenada)

### Análisis de correlación

La matriz de correlación muestra que no existen relaciones fuertes entre la mayoría de las variables predictoras, lo que indica ausencia de multicolinealidad relevante y un conjunto de features bien balanceado.

Al analizar la correlación con el target (`zona_caliente`), se observa que las variables espaciales (`long_std` y `lat_std`) presentan la mayor asociación. Este resultado es coherente con la definición del objetivo del modelo, ya que la clase “zona caliente” se construye a partir de la densidad espacial de incidentes en una grilla geográfica.

Las variables temporales (`hora_sin`, `hora_cos`, `dia_semana`, `mes`, `es_fin_de_semana`) muestran correlaciones más bajas, lo cual es esperado, dado que su rol es aportar **contexto temporal** y no definir el target. Estas variables permiten al modelo aprender **en qué momentos ciertas zonas tienden a concentrar más eventos**, complementando la información espacial.

La correlación elevada entre `dia_semana` y `es_fin_de_semana` es esperable, ya que una variable deriva de la otra, y no representa un problema para el modelado.





## Conclusión de Feature Engineering

La ingeniería de características permitió transformar las variables originales en una representación más adecuada para el modelado del fenómeno delictivo. La incorporación de variables temporales (`mes`, `dia_semana`, `es_fin_de_semana`) y la codificación cíclica de la hora (`hora_sin`, `hora_cos`) permitió capturar el contexto temporal sin distorsionar la naturaleza continua del tiempo.

En el componente espacial, la estandarización de las coordenadas (`lat_std`, `long_std`) aseguró una correcta escala de las variables geográficas, mejorando la estabilidad y el desempeño del modelo. Dado que el objetivo se define como **zona caliente (hotspot espacial)**, la ubicación constituye la principal fuente de señal, mientras que las variables temporales aportan información complementaria sobre cuándo ciertas zonas tienden a concentrar más incidentes.

El análisis de correlación confirmó que las variables seleccionadas no presentan multicolinealidad relevante y aportan información desde distintas dimensiones, validando el conjunto final de features. En conjunto, estas transformaciones permiten pasar de un dataset descriptivo a una estructura optimizada para Machine Learning, alineada con un enfoque geoespacial y orientada a la identificación de zonas de riesgo.


## Definición de la Variable Objetivo (Target)

Dado que el dataset original no cuenta con una variable objetivo predefinida para clasificar, se procedió a construir una mediante **Feature Engineering**. Estamos frente a un problema de **Clasificación Binaria**.

La variable objetivo creada se llama `zona_caliente`.

### ¿Cómo se calcula?
Se agruparon las coordenadas geográficas en zonas (`zona_lat` + `zona_long`) y se calculó la densidad de delitos por zona **exclusivamente en el conjunto de entrenamiento**, esto con el fin de evitar la fuga de información (*data leakage*).

Las clases se definen de la siguiente manera:

* **Clase 1 (Zona Caliente):** Zonas cuya densidad de delitos se encuentra en el percentil 75 o superior. Representan las áreas de mayor riesgo.
* **Clase 0 (Zona No Caliente):** Zonas con una densidad de delitos inferior al percentil 75.

> **Objetivo del modelo:** De esta forma, el algoritmo aprenderá a predecir la probabilidad de que un evento, dado su contexto temporal y espacial, ocurra en una zona de alta concentración delictiva.

#MODELADO
## Modelado de Machine Learning

Una vez preparado el dataset, se procede a aplicar modelos de Machine Learning.

El objetivo es construir modelos capaces de clasificar eventos delictivos en:

- 0 → zona de baja concentración
- 1 → zona de alta concentración (zona caliente)

Se utilizarán dos tipos de modelos:

1. Regresión Logística (modelo lineal base)
2. Random Forest (modelo no lineal)

Esto permitirá comparar enfoques y comprender mejor la naturaleza del problema.

## División de los datos

Se divide el dataset en:

- entrenamiento (80%)
- prueba (20%)

Se utiliza división estratificada para mantener la proporción del target.

## Modelo 1: Regresión Logística

Se utiliza regresión logística como modelo base.

Este modelo:

- es lineal
- fácil de interpretar
- devuelve probabilidades

Se aplica `class_weight="balanced"` para compensar posibles desbalances.

In [ ]:
# Asegurar que no haya strings
X_train = X_train.drop(columns=['zona'], errors='ignore')
X_test = X_test.drop(columns=['zona'], errors='ignore')

# Modelo
from sklearn.linear_model import LogisticRegression

modelo_lr = LogisticRegression(
    max_iter=2000,
    class_weight='balanced',
    random_state=42
)

modelo_lr.fit(X_train, y_train)

In [ ]:
# =========================
# PREDICCIONES DEL MODELO
# =========================

# Predicción de clases (0 o 1)
y_pred = modelo_lr.predict(X_test)

# Predicción de probabilidades
y_proba = modelo_lr.predict_proba(X_test)

# Ver resultados
print("Primeras clases reales:")
print(y_test.values[:10])

print("\nPrimeras predicciones:")
print(y_pred[:10])

# Mostrar probabilidades (.DataFrame más claro)
import pandas as pd

proba_df = pd.DataFrame(
    y_proba[:10],
    columns=["No caliente (0)", "Caliente (1)"]
)

print("\nProbabilidades:")
print(proba_df)


## Predicciones del Modelo

Una vez entrenado el modelo, se generan predicciones sobre el conjunto de prueba, obteniendo tanto la clase estimada como la probabilidad asociada a cada evento.

La Regresión Logística no solo clasifica los eventos en zonas de alta o baja concentración delictiva, sino que también estima la **probabilidad de pertenencia a una zona caliente (hotspot espacial)**, considerando de forma conjunta la ubicación del evento y su contexto temporal.

Valores de probabilidad elevados indican que el modelo reconoce patrones espaciales y temporales fuertemente asociados a zonas de alta concentración, reflejando una mayor confianza en la predicción. En cambio, probabilidades cercanas al umbral corresponden a casos más ambiguos, donde la decisión es más sensible al criterio de clasificación.

Estas probabilidades resultan especialmente útiles para analizar casos límite y para ajustar posteriormente el umbral de decisión según el objetivo operativo, sin modificar el modelo entrenado.

In [ ]:


from sklearn.metrics import accuracy_score, classification_report

# =========================
# ANÁLISIS DE UMBRALES
# =========================

y_test_ref = y_test.copy()
y_proba_ref = y_proba[:, 1].copy()

umbrales = [0.30, 0.40, 0.45, 0.50]

for umbral in umbrales:
    y_pred_umbral = (y_proba_ref >= umbral).astype(int)

    print("=" * 40)
    print(f"Umbral evaluado: {umbral}")
    print("Accuracy:", round(accuracy_score(y_test_ref, y_pred_umbral), 3))
    print(classification_report(y_test_ref, y_pred_umbral))




### Análisis manual del umbral y criterio de seguridad

El análisis de umbrales muestra cómo varía el desempeño del modelo al modificar el punto de corte sobre las probabilidades predichas, sin alterar el modelo entrenado ni el pipeline de datos.

Con umbrales más bajos (por ejemplo, 0.30 y 0.40), el modelo incrementa notablemente el **recall de la clase “zona caliente”**, llegando a detectar casi la totalidad de las zonas de alta concentración. Esto reduce de forma significativa los **falsos negativos**, a costa de un aumento en los falsos positivos y una menor precisión.

En particular, el umbral **0.40** presenta un equilibrio operativo adecuado para un contexto de **seguridad urbana**, con un recall cercano al 97%, lo que implica que el modelo reconoce con alta sensibilidad las zonas de riesgo, minimizando omisiones críticas.

No obstante, para el **modelo base** se mantiene el umbral estándar de **0.50**, ya que ofrece un mejor balance global entre *precision*, *recall* y *accuracy*, y permite una evaluación general del desempeño sin priorizar un escenario operativo específico.

De este modo, el ajuste del umbral se presenta como una herramienta para adaptar el mismo modelo a distintos criterios de decisión, manteniendo como objetivo central la identificación de **zonas calientes en función del contexto geoespacial y temporal**.


## Matriz de confusión

Permite analizar los aciertos y errores del modelo.


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

matriz = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(matriz)
disp.plot()
plt.title("Matriz de Confusión - Regresión Logística")
plt.show()

VN, FP, FN, VP = matriz.ravel()

print("VN:", VN)
print("FP:", FP)
print("FN:", FN)
print("VP:", VP)

### Interpretación de la matriz de confusión

La matriz de confusión corresponde al modelo base de Regresión Logística utilizando un umbral de decisión de 0.50. Los resultados muestran que el modelo identifica correctamente una gran cantidad de zonas no calientes (VN) y detecta la mayoría de las zonas calientes (VP).

El número reducido de falsos negativos (FN) indica que el modelo omite pocas zonas de alta concentración delictiva, lo cual es especialmente relevante en un contexto de seguridad urbana, donde este tipo de error resulta crítico. Por el contrario, los falsos positivos (FP) representan alertas adicionales en zonas no críticas, un costo operativo aceptable frente al riesgo de no detectar áreas peligrosas.

En conjunto, la matriz de confusión confirma que el modelo presenta un comportamiento equilibrado, con buena capacidad para discriminar zonas calientes, manteniendo un enfoque conservador orientado a la detección de riesgo geoespacial.

## Evaluación del Modelo de Regresión Logística

Al ajustar el umbral de decisión a 0.40, el modelo de Regresión Logística alcanza un accuracy de aproximadamente 0.80, acompañado de un recall elevado para la clase “zona caliente”.

El análisis de la matriz de confusión muestra que el modelo logra detectar la gran mayoría de las zonas calientes, reduciendo significativamente los falsos negativos, aunque a costa de un incremento en los falsos positivos.

Las métricas obtenidas para la clase de interés (zona caliente) son:
- Precision: 0.56  
- Recall: 0.97  
- F1-score: 0.71  

El alto valor de recall indica que el modelo identifica casi la totalidad de las zonas de alta concentración delictiva, lo cual resulta prioritario en un contexto de seguridad urbana. La elección del umbral 0.40 responde a la necesidad de minimizar omisiones de zonas de riesgo, aceptando falsas alarmas como un costo operativo razonable.

No obstante, para el análisis general del desempeño se mantiene el umbral estándar de 0.50 como referencia base, ya que ofrece un equilibrio más estable entre precisión, recall y accuracy.



## Modelo 2: Random Forest

Se utiliza un modelo no lineal para capturar patrones complejos.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd

# =========================
# MODELO RANDOM FOREST
# =========================

modelo_rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=5,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42
)

# =========================
# ENTRENAMIENTO
# =========================

modelo_rf.fit(X_train, y_train)

# =========================
# PREDICCIONES TEST
# =========================

y_pred_rf = modelo_rf.predict(X_test)

print("- Accuracy Test:", round(accuracy_score(y_test, y_pred_rf), 3))

print("\nReporte TEST:")
print(classification_report(y_test, y_pred_rf))

# =========================
# VALIDACIÓN SOBREAJUSTE
# =========================

y_train_pred_rf = modelo_rf.predict(X_train)

print("\n- Accuracy Train:", round(accuracy_score(y_train, y_train_pred_rf), 3))

# =========================
# FEATURE IMPORTANCE (CORREGIDO)
# =========================

importancias = modelo_rf.feature_importances_

df_importancia = pd.DataFrame({
    'feature': X_train.columns,
    'importancia': importancias
}).sort_values(by='importancia', ascending=False)

print("\n- Importancia de variables:")
print(df_importancia)

In [ ]:
# =========================
# VALIDACIÓN DEL MODELO
# =========================

from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Predicciones
y_pred_final = y_pred_rf
# =========================
# REPORTE
# =========================

print("Reporte completo:\n")
print(classification_report(y_test, y_pred_final))


# =========================
# MATRIZ DE CONFUSIÓN
# =========================

matriz = confusion_matrix(y_test, y_pred_final)

disp = ConfusionMatrixDisplay(confusion_matrix=matriz)
disp.plot(cmap="viridis")

plt.title("Matriz de Confusión")
plt.show()

> **💡 Observación sobre la visualización:** > En la matriz de confusión, el valor del cuadrante superior izquierdo (Verdaderos Negativos) se muestra como **`2e+04`**. Esto corresponde a la notación científica estándar de la librería gráfica para representar el número exacto **20.299** ($2 \times 10^4$).
>
> Se decidió mantener este formato automático por defecto para preservar la legibilidad del cuadro, ya que el alto volumen de casos clasificados correctamente en la clase 0 (zonas no calientes) es significativamente mayor al resto debido al umbral del percentil 75 utilizado para definir el target.

## Evaluación del Modelo Random Forest

El modelo Random Forest obtiene un accuracy cercano al 0.90 en el conjunto de prueba, mostrando un desempeño elevado y estable. En la detección de zonas calientes, alcanza un recall de 0.95, lo que indica que identifica la gran mayoría de las áreas de alta concentración delictiva.

La matriz de confusión muestra un número muy reducido de falsos negativos, lo cual resulta especialmente relevante en un contexto de seguridad urbana, donde la omisión de zonas de riesgo representa el error más crítico. Si bien existen falsos positivos, estos se consideran un costo operativo aceptable frente al beneficio de una alta capacidad de detección.

En conjunto, los resultados confirman que Random Forest captura de manera efectiva patrones espaciales complejos, superando al modelo lineal base y consolidándose como una alternativa robusta para la identificación de hotspots geoespaciales condicionados por el contexto temporal.


## Comparación entre modelos

Se comparan ambos modelos para evaluar desempeño.

In [ ]:
# ==========================================================
# COMPARACIÓN DE MÉTRICAS (MODELOS BASE)
# ==========================================================

from sklearn.metrics import accuracy_score, precision_score, recall_score
import matplotlib.pyplot as plt

# -------------------------
# Copias de seguridad
# -------------------------
y_test_ref = y_test.copy()

y_pred_lr_ref = y_pred.copy()       # Regresión Logística (umbral base 0.50)
y_pred_rf_ref = y_pred_rf.copy()    # Random Forest

# -------------------------
# MÉTRICAS REGRESIÓN LOGÍSTICA
# -------------------------
acc_lr = accuracy_score(y_test_ref, y_pred_lr_ref)
prec_lr = precision_score(y_test_ref, y_pred_lr_ref)
rec_lr = recall_score(y_test_ref, y_pred_lr_ref)

# -------------------------
# MÉTRICAS RANDOM FOREST
# -------------------------
acc_rf = accuracy_score(y_test_ref, y_pred_rf_ref)
prec_rf = precision_score(y_test_ref, y_pred_rf_ref)
rec_rf = recall_score(y_test_ref, y_pred_rf_ref)

# -------------------------
# MOSTRAR MÉTRICAS
# -------------------------
print("=== REGRESIÓN LOGÍSTICA ===")
print(f"Accuracy : {acc_lr:.3f}")
print(f"Precision: {prec_lr:.3f}")
print(f"Recall   : {rec_lr:.3f}")

print("\n=== RANDOM FOREST ===")
print(f"Accuracy : {acc_rf:.3f}")
print(f"Precision: {prec_rf:.3f}")
print(f"Recall   : {rec_rf:.3f}")

# -------------------------
# GRÁFICO COMPARATIVO
# -------------------------
modelos = ['Regresión Logística', 'Random Forest']
accuracy = [acc_lr, acc_rf]
precision = [prec_lr, prec_rf]
recall = [rec_lr, rec_rf]

x = range(len(modelos))

plt.figure(figsize=(7, 4))

plt.plot(x, accuracy, marker='o', label='Accuracy')
plt.plot(x, precision, marker='o', label='Precision (clase 1)')
plt.plot(x, recall, marker='o', label='Recall (clase 1)')

plt.xticks(x, modelos)
plt.ylim(0, 1.05)

plt.title('Comparación de Métricas entre Modelos (Escenario Base)')
plt.xlabel('Modelo')
plt.ylabel('Valor')

plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


## Comparación de modelos

La comparación de métricas evidencia que el modelo Random Forest supera a la Regresión Logística en el desempeño general para la identificación de zonas calientes.

En términos de accuracy, Random Forest alcanza un valor cercano al 0.90, frente a aproximadamente 0.82 obtenido por la Regresión Logística. Asimismo, se observa una mejora en la precisión de la clase “zona caliente”, que pasa de 0.60 en el modelo lineal a 0.74 en Random Forest, reduciendo la cantidad de falsas alarmas.

El recall también se incrementa, lo que indica una mayor capacidad del modelo no lineal para detectar zonas de alta concentración delictiva. Esta mejora es especialmente relevante en un contexto de seguridad urbana, donde la omisión de zonas de riesgo representa el error más crítico.

En conjunto, los resultados confirman que Random Forest captura de forma más efectiva patrones espaciales complejos que el modelo lineal, consolidándose como la alternativa más robusta para la detección de hotspots geoespaciales condicionados por el contexto temporal.


In [ ]:
# ==========================================================
# CROSS-VALIDATION POR FOLDS (SIN DATA LEAKAGE)
# ==========================================================

from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score
from sklearn.base import clone
import numpy as np
import pandas as pd

# -------------------------
# Construir dataset auxiliar PARA CV
# (features + zona alineadas)
# -------------------------
X_all = pd.concat([X_train, X_test], axis=0).reset_index(drop=True)

zonas_all = pd.concat(
    [
        df.loc[X_train.index, 'zona'],
        df.loc[X_test.index, 'zona']
    ],
    axis=0
).reset_index(drop=True)

# Features del modelo
cols_modelo = [
    'lat_std', 'long_std',
    'hora_sin', 'hora_cos',
    'mes', 'dia_semana', 'es_fin_de_semana'
]

kf = KFold(n_splits=5, shuffle=True, random_state=42)

scores_lr = []
scores_rf = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X_all), 1):

    X_tr = X_all.iloc[train_idx].copy()
    X_va = X_all.iloc[val_idx].copy()

    zonas_tr = zonas_all.iloc[train_idx]
    zonas_va = zonas_all.iloc[val_idx]

    # Densidad SOLO con el train del fold
    dens_tr = zonas_tr.value_counts()

    densidad_tr = zonas_tr.map(dens_tr)
    densidad_va = zonas_va.map(dens_tr).fillna(0)

    umbral_fold = densidad_tr.quantile(0.75)

    y_tr = (densidad_tr >= umbral_fold).astype(int)
    y_va = (densidad_va >= umbral_fold).astype(int)

    # Clonar modelos (clave para no contaminar folds)
    lr_fold = clone(modelo_lr)
    rf_fold = clone(modelo_rf)

    lr_fold.fit(X_tr[cols_modelo], y_tr)
    rf_fold.fit(X_tr[cols_modelo], y_tr)

    pred_lr = lr_fold.predict(X_va[cols_modelo])
    pred_rf = rf_fold.predict(X_va[cols_modelo])

    scores_lr.append(accuracy_score(y_va, pred_lr))
    scores_rf.append(accuracy_score(y_va, pred_rf))

# -------------------------
# Resultados
# -------------------------
scores_lr = np.array(scores_lr)
scores_rf = np.array(scores_rf)

print("=== REGRESIÓN LOGÍSTICA (CV sin leakage) ===")
print("Scores:", scores_lr)
print("Promedio:", round(scores_lr.mean(), 3))
print("Desviación:", round(scores_lr.std(), 3))

print("\n=== RANDOM FOREST (CV sin leakage) ===")
print("Scores:", scores_rf)
print("Promedio:", round(scores_rf.mean(), 3))
print("Desviación:", round(scores_rf.std(), 3))


In [ ]:
# ==========================================================
# GRÁFICO DE VARIABILIDAD POR FOLD (CV SIN LEAKAGE)
# ==========================================================

folds = np.arange(1, len(scores_lr) + 1)

plt.figure(figsize=(7, 4))

# Regresión Logística
plt.plot(
    folds,
    scores_lr,
    marker='o',
    label='Regresión Logística'
)

# Random Forest
plt.plot(
    folds,
    scores_rf,
    marker='o',
    label='Random Forest'
)

# Líneas de promedio
plt.axhline(scores_lr.mean(), linestyle='--', alpha=0.6)
plt.axhline(scores_rf.mean(), linestyle='--', alpha=0.6)

plt.title('Variabilidad del Desempeño por Fold (CV sin leakage)')
plt.xlabel('Fold')
plt.ylabel('Accuracy')

plt.xticks(folds)
plt.ylim(0.78, 0.93)

plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## Curva ROC y métrica AUC

Además de las métricas de clasificación tradicionales, utilizaremos la **Curva ROC** y el valor **AUC (Área Bajo la Curva)** para comparar la capacidad general de los modelos al separar las clases.

La curva ROC grafica la tasa de verdaderos positivos frente a la tasa de falsos positivos en distintos umbrales de decisión. Cuanto más cerca de 1 esté el AUC, mejor será la capacidad del modelo para distinguir entre zonas calientes y zonas de bajo riesgo.

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

# Obtenemos las probabilidades de la clase positiva (zona caliente = 1) para cada modelo
y_prob_lr = modelo_lr.predict_proba(X_test)[:, 1]
y_prob_rf = modelo_rf.predict_proba(X_test)[:, 1]

# Calculamos la curva ROC y el AUC para Regresión Logística
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)
auc_lr = roc_auc_score(y_test, y_prob_lr)

# Calculamos la curva ROC y el AUC para Random Forest
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)
auc_rf = roc_auc_score(y_test, y_prob_rf)

# Graficamos ambas curvas ROC
plt.figure(figsize=(8, 6))

plt.plot(
    fpr_lr,
    tpr_lr,
    label=f"Regresión Logística - AUC = {auc_lr:.3f}",
    color='steelblue'
)

plt.plot(
    fpr_rf,
    tpr_rf,
    label=f"Random Forest - AUC = {auc_rf:.3f}",
    color='darkorange'
)

# Línea de referencia de un modelo aleatorio
plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    color="gray",
    label="Modelo aleatorio"
)

plt.title("Curva ROC - Comparación de Modelos")
plt.xlabel("Tasa de Falsos Positivos")
plt.ylabel("Tasa de Verdaderos Positivos")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()

## Validación Cruzada de los Modelos

Se aplicó validación cruzada sin fuga de información con el objetivo de evaluar la estabilidad y capacidad de generalización de los modelos bajo distintas particiones del dataset.

Los resultados muestran que la Regresión Logística presenta un desempeño estable, con variaciones moderadas entre folds, lo que confirma su rol como modelo base consistente. Por su parte, el modelo Random Forest alcanza un rendimiento promedio superior y exhibe una variabilidad aún menor entre folds, indicando una alta estabilidad además de una mayor capacidad predictiva.

El gráfico de desempeño por fold refuerza estas conclusiones, mostrando que Random Forest mantiene valores de accuracy elevados y homogéneos en todas las particiones. En conjunto, la validación cruzada confirma que Random Forest no solo mejora el desempeño respecto al modelo lineal, sino que también generaliza de manera robusta, siendo adecuado para la detección de zonas calientes en un enfoque de seguridad urbana.


## Predicción de escenarios futuros

Se simulan condiciones para predecir riesgo delictivo.

In [ ]:
# ==========================================================
# SIMULACIÓN DE ESCENARIO (INTERPRETACIÓN DEL MODELO)
# ==========================================================

# Escenario hipotético
hora = 22
dia = 5
mes = 12

lat_real = -34.60
lon_real = -58.38

# Construcción del vector de entrada
escenario_dict = {}

for col in X_train.columns:

    if col == 'lat_std':
        escenario_dict[col] = (lat_real - X_train['lat_std'].mean()) / X_train['lat_std'].std()

    elif col == 'long_std':
        escenario_dict[col] = (lon_real - X_train['long_std'].mean()) / X_train['long_std'].std()

    elif col == 'hora_sin':
        escenario_dict[col] = np.sin(2 * np.pi * hora / 24)

    elif col == 'hora_cos':
        escenario_dict[col] = np.cos(2 * np.pi * hora / 24)

    elif col == 'mes':
        escenario_dict[col] = mes

    elif col == 'dia_semana':
        escenario_dict[col] = dia

    elif col == 'es_fin_de_semana':
        escenario_dict[col] = 1 if dia >= 5 else 0

    else:
        escenario_dict[col] = 0

escenario = pd.DataFrame([escenario_dict])

# Predicciones (interpretación, no predicción futura)
proba_lr = modelo_lr.predict_proba(escenario)[0][1]
proba_rf = modelo_rf.predict_proba(escenario)[0][1]

print("=== SIMULACIÓN DE ESCENARIO ===")
print(f"Ubicación: ({lat_real}, {lon_real})")
print(f"Contexto temporal: hora {hora}, día {dia}, mes {mes}")

print("\nProbabilidad de pertenencia a zona caliente (según el modelo):")
print(f"Regresión Logística: {proba_lr:.2f}")
print(f"Random Forest: {proba_rf:.2f}")


## Guardado del modelo final

Tras evaluar el rendimiento, la estabilidad en validación cruzada y la capacidad de captura de patrones geoespaciales, seleccionamos **Random Forest** como el modelo final de nuestro proyecto.

Para poder llevar este modelo a un entorno de producción o reutilizarlo en el futuro sin tener que entrenarlo nuevamente, guardaremos el objeto del modelo usando la librería `joblib`.

*Nota importante:* Como nuestro modelo requiere que las coordenadas geográficas estén estandarizadas, también debemos guardar el objeto `StandardScaler` (`scaler`) para aplicarlo sobre nuevos datos antes de predecir.

In [ ]:
import joblib

# Definimos los nombres de los archivos
archivo_modelo = "modelo_rf_seguridad_caba.pkl"
archivo_scaler = "scaler_coordenadas_caba.pkl"

# Guardamos el modelo Random Forest y el Escalador
joblib.dump(modelo_rf, archivo_modelo)
joblib.dump(scaler, archivo_scaler)

print(f"✅ Modelo guardado exitosamente como: {archivo_modelo}")
print(f"✅ Escalador guardado exitosamente como: {archivo_scaler}")

## Limitaciones y posibles mejoras

A pesar de los buenos resultados obtenidos, especialmente con el modelo Random Forest, es fundamental reconocer las limitaciones del proyecto para dimensionar correctamente su alcance metodológico:

1. **Datos Faltantes Eliminados:** Se debió descartar aproximadamente un 2.4% del dataset original debido a coordenadas nulas o erróneas. Aunque la proporción es baja, en un contexto de seguridad urbana, esos registros podrían concentrar información de zonas específicas con deficiencias en la carga de datos policiales.
2. **Definición Estática de Zonas:** La grilla espacial utilizada para definir las zonas es una simplificación geométrica. En la realidad, las fronteras de los barrios, la iluminación o la presencia de avenidas cortan el tejido urbano de formas irregulares que el modelo actual no contempla geométricamente.
3. **Ausencia de Variables Exógenas:** El modelo se apoya enteramente en el contexto espacio-temporal (días, horas, ubicación). No se cuenta con datos de clima, densidad poblacional flotante, eventos masivos (partidos de fútbol, recitales) ni ubicación de cámaras de seguridad, factores que impactan fuertemente en la criminalidad.

**Mejoras para el futuro:**
* Integrar bases de datos externas (como clima o calendario de feriados/eventos).
* Probar modelos de ensamble más avanzados como *XGBoost* o *LightGBM*.
* En lugar de una clasificación binaria espacial, intentar un enfoque de series temporales (ej. LSTM) para predecir no solo *dónde*, sino el volumen de delitos esperado en la próxima hora.



#CONCLUSIÓN DEL ESCENARIO

Interpretación de la simulación de escenario

Las probabilidades obtenidas en la simulación no representan una predicción de riesgo futuro ni una evaluación operativa de seguridad. Indican la probabilidad de que un evento con características similares haya ocurrido históricamente en una zona caliente, según los patrones aprendidos durante el entrenamiento.

Valores cercanos a cero reflejan que la combinación de ubicación y contexto temporal no se asemeja a los eventos asociados a zonas de alta concentración delictiva en los datos históricos. Este comportamiento es consistente con un modelo de clasificación geoespacial y refuerza la necesidad de interpretar las salidas dentro del dominio de entrenamiento y del alcance metodológico del modelo.


## Análisis Final del Proyecto

El presente trabajo abordó el análisis del fenómeno delictivo en la Ciudad de Buenos Aires mediante un enfoque integral que combina análisis exploratorio de datos (EDA), ingeniería de características y modelos de Machine Learning, con un énfasis explícito en la dimensión geoespacial del problema.

### Análisis Exploratorio de Datos (EDA)

El análisis exploratorio permitió identificar patrones espaciales y temporales relevantes. Se observó una fuerte concentración de delitos en determinadas zonas de la ciudad, así como variaciones significativas según la franja horaria, el día de la semana y el tipo de jornada. Estos patrones evidencian que el fenómeno delictivo no se distribuye de manera homogénea ni en el espacio ni en el tiempo, y justificaron la necesidad de un enfoque geoespacial.

Los resultados del EDA fueron fundamentales para comprender la dinámica del problema y orientar el diseño del target y de las variables explicativas utilizadas en el modelado.

### Ingeniería de Características

A partir del EDA, se aplicaron técnicas de feature engineering para representar adecuadamente la información espacial y temporal. Se incorporó la transformación cíclica de la variable horaria, la creación de variables temporales (mes, día de la semana y fin de semana) y la estandarización de las coordenadas geográficas.

El objetivo del modelo se definió como la identificación de **zonas calientes (hotspots espaciales)**, construidas a partir de la densidad relativa de eventos en una grilla geográfica. Las variables temporales se utilizaron como contexto explicativo, mientras que la ubicación geográfica constituyó la principal fuente de señal, coherente con la definición del target.

### Modelado y Enfoque Metodológico

Se implementaron dos modelos de clasificación:

- **Regresión Logística**, utilizada como modelo base por su interpretabilidad y estabilidad.
- **Random Forest**, empleado para capturar relaciones no lineales y patrones espaciales complejos.

Durante el proceso se aplicaron buenas prácticas de Machine Learning, incluyendo la separación train/test, el control de fuga de información en la definición del target y la evaluación con múltiples métricas. Asimismo, se analizó el impacto del umbral de decisión en la Regresión Logística para explorar escenarios operativos orientados a seguridad.

### Evaluación y Validación

En el escenario base, la Regresión Logística alcanzó un accuracy cercano a 0.82, con un recall elevado para la clase “zona caliente”, lo que confirma su capacidad para detectar la mayoría de las áreas de alta concentración delictiva, aunque con menor precisión.

Por su parte, el modelo Random Forest obtuvo un desempeño superior, con un accuracy cercano a 0.90, mayor precisión y un recall aún más alto. Esto indica una mejor capacidad para identificar zonas calientes reduciendo falsos negativos, aspecto crítico en un contexto de seguridad urbana.

La validación cruzada sin fuga de información confirmó estas conclusiones. Random Forest no solo presentó un mayor rendimiento promedio, sino también una baja variabilidad entre folds, lo que evidencia una buena capacidad de generalización. La Regresión Logística, si bien más simple, mostró un comportamiento estable y consistente, validando su rol como modelo base.

### Simulación de Escenarios e Interpretación

Se realizaron simulaciones ilustrativas para analizar cómo los modelos responden ante eventos con una determinada ubicación y contexto temporal. Es importante destacar que estas simulaciones no representan predicciones de riesgo futuro, sino una **medida de similitud con eventos históricos ocurridos en zonas calientes**, según los patrones aprendidos durante el entrenamiento.

Valores bajos de probabilidad indican que la combinación espacial y temporal evaluada no se asemeja a los eventos históricamente asociados a zonas de alta concentración delictiva, dentro del dominio de entrenamiento del modelo (Ciudad de Buenos Aires).

### Alcance, Limitaciones y Trabajo Futuro

El enfoque propuesto permite identificar y analizar patrones históricos de concentración delictiva, funcionando como una herramienta de apoyo analítico para la comprensión del fenómeno y la planificación basada en evidencia. No obstante, el modelo no realiza predicciones temporales futuras ni estimaciones operativas en tiempo real.

Como líneas de trabajo futuro, se propone incorporar validaciones temporales, modelado explícito por ventanas de tiempo, y variables externas (movilidad, clima, eventos urbanos), con el objetivo de evolucionar hacia modelos predictivos espacio-temporales más completos.

### Impacto y Aplicación en Seguridad Urbana

El enfoque desarrollado tiene potencial de aplicación en organismos de seguridad y planificación urbana como herramienta de análisis y apoyo a la toma de decisiones. Permite identificar zonas históricamente críticas, comparar contextos espaciales y temporales, y optimizar la asignación de recursos de manera informada.

En conjunto, el proyecto demuestra cómo el uso combinado de análisis geoespacial y Machine Learning puede contribuir a una mejor comprensión del comportamiento delictivo, respetando el alcance metodológico del modelo y evitando interpretaciones incorrectas de sus resultados.


## Anexo final y referencias

**Dataset utilizado:**
El conjunto de datos base para este proyecto fue extraído del portal de datos abiertos del Gobierno de la Ciudad de Buenos Aires:
* [Delitos 2024 – Gobierno de la Ciudad de Buenos Aires](https://data.buenosaires.gob.ar/de/dataset/delitos/resource/49f58c2e-21d7-4766-84e0-4bb753d28478)

**Bibliografía y Herramientas Consultadas:**
* **Scikit-Learn Documentation:** Utilizado para la implementación de las métricas de evaluación, validación cruzada estructurada y modelos de clasificación.
* **Folium:** Para la renderización de mapas de calor interactivos y visualización de clústeres espaciales.
* **Géron, A. (2019).** *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow.* O'Reilly Media. (Referencia teórica para las transformaciones de *Feature Engineering* y evaluación mediante Curvas ROC).

## 🤝 Agradecimientos

Este proyecto ha sido una instancia fundamental en mi formación, y me gustaría expresar mi sincero agradecimiento a quienes lo hicieron posible:

* **Al Profesor Ariel Palazzesi**, por su guía, paciencia y dedicación en cada clase, transmitiendo no solo conocimientos técnicos de *Machine Learning*, sino también el criterio necesario para abordar proyectos con impacto real.
* **A Talento Tech**, por el compromiso con la formación en habilidades digitales y por generar espacios de aprendizaje tan valiosos para nuestra comunidad.
* **A la Agencia de Habilidades de la Ciudad de Buenos Aires y al Gobierno de la Ciudad de Buenos Aires**, por acercar estas oportunidades de capacitación tecnológica, fundamentales para el desarrollo profesional y la innovación en el ámbito público.

Fue un desafío enriquecedor aplicar estas herramientas al análisis geoespacial de nuestra ciudad. ¡Muchas gracias por el acompañamiento!